# LARA-Q2 on DAS - patched reconstruction and 2D images

Six LARA-Q2 checkpoints (`lambda = 1e-4`, STEAD-trained) are applied to the DAS
fibre-optic records at CR = 2, 5, 10, 20, 50 and 100.

Each 2000-sample channel is split into 1500-sample patches with 50% overlap,
every patch is encoded and decoded independently through the quantiser, and the
patches are recombined with Gaussian weighted overlap-add.  The result is a
reconstructed (channels x time) shot gather, shown next to the original and the
residual for every CR.

No training and no new validation: the checkpoints and the bit-accurate rates
come from the completed run in `das_validation_26102026`.

## Configuration

Paths can be overridden with environment variables:

* `LARA_DAS_DIR` - folder holding `out_eq_{1,44,74}_pre.mat`
* `LARA_RESULTS_DIR` - folder holding `ckpts_baselines/`
* `LARA_CKPT_DIR` - override the checkpoint folder directly
* `LARA_DAS_EVENT` - event shown in the figures (default 44)
* `LARA_Q2_OUT` - output folder (default `q2_das_validation`)

In [ ]:
"""Configuration for the LARA-Q2 DAS visualisation (27 10 2026)."""
import os
import sys
import json
import math
import glob
import time
import csv
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------ model geometry
INPUT_LEN = 1500
WINDOW_START = 0
LARA_VARIANT = "pyramid_funnel32"
BASELINE_NAMES = ["GeneralizedAutoencoder", "AE_PureConcat"]
ARCH2_SERIES = ["LARA-Q2"]
WIDTHS = [750, 500, 300, 150, 100, 75, 50, 30, 25, 15]
LAMBDAS = [1e-5, 1e-4, 1e-3]
LAM_HEADLINE = 1e-4

# ------------------------------------------------------------ what to run
CRS = [2, 5, 10, 20, 50, 100]
DEPTH = 10                 # target bits per symbol used for every CR

# ------------------------------------------------------------ DAS
DAS_EVENT = int(os.environ.get("LARA_DAS_EVENT", "44"))
DAS_EVENTS_ALL = [1, 44, 74]
DAS_KEY = "outF"
DAS_EXPECTED_TRACES = 960
DAS_DIR = os.environ.get("LARA_DAS_DIR", "data")

N_CALIB_TARGET = 2000
SPLIT_STRIDE = 10
CALIB_RESIDUES = tuple(range(8))
EVAL_RESIDUES = (9,)

# ------------------------------------------------------------ patching
PATCH_OVERLAP = 750
GAUSS_SIGMA_FRACTION = 0.25
GAUSS_FLOOR = 1e-3
BATCH = 64
DISPLAY_PERCENTILE = 99.0
SAVE_NPZ = True
# 0 = all channels of the event; a positive value shows only the first N
MAX_CHANNELS = int(os.environ.get("LARA_Q2_CHANNELS", "0"))


try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:                 # a notebook kernel has no __file__
    HERE = os.path.abspath(os.getcwd())


def _find_results_dir():
    cand = os.environ.get("LARA_RESULTS_DIR", "")
    if cand:
        return os.path.abspath(cand)
    here = HERE
    for _ in range(6):
        for probe in (os.path.join(here, "ckpts_baselines"),
                      os.path.join(here, "results", "ckpts_baselines")):
            if os.path.isdir(probe):
                return os.path.dirname(probe)
        here = os.path.dirname(here)
    return os.path.abspath(".")


RESULTS_DIR = _find_results_dir()
CKPT_DIR = os.environ.get("LARA_CKPT_DIR",
                          os.path.join(RESULTS_DIR, "ckpts_baselines"))
OUT_DIR = os.environ.get("LARA_Q2_OUT", "q2_das_validation")
FIG_DIR = os.path.join(OUT_DIR, "figures")


def _find_val_dir():
    """The completed DAS validation run, wherever it sits."""
    cand = os.environ.get("LARA_VAL_DIR", "")
    if cand and os.path.isdir(cand):
        return os.path.abspath(cand)
    probes = [
        os.path.join(HERE, "das_validation_26102026",
                     "das_validation_26102026"),
        os.path.join(os.getcwd(), "das_validation_26102026",
                     "das_validation_26102026"),
        os.path.join(RESULTS_DIR, "das_validation_26102026",
                     "das_validation_26102026"),
        os.path.join(os.path.dirname(RESULTS_DIR), "DAS_Validation",
                     "das_validation_26102026", "das_validation_26102026"),
    ]
    for probe in probes:
        if os.path.isdir(probe):
            return os.path.abspath(probe)
    for root in (HERE, RESULTS_DIR):
        hits = sorted(glob.glob(os.path.join(
            root, "**", "das_validation_*", "das_validation_*"),
            recursive=True))
        if hits:
            return os.path.abspath(hits[0])
    return ""


VAL_DIR = _find_val_dir()
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("LARA-Q2 on DAS - patched reconstruction")
print("-" * 60)
print(f"device      {DEVICE}")
if DEVICE.type == "cuda":
    print(f"gpu         {torch.cuda.get_device_name(0)}")
print(f"das dir     {os.path.abspath(DAS_DIR)}")
print(f"ckpt dir    {os.path.abspath(CKPT_DIR)}")
print(f"results dir {os.path.abspath(RESULTS_DIR)}")
print(f"out dir     {os.path.abspath(OUT_DIR)}")
print(f"validation  {VAL_DIR or 'not found (rates fall back to nominal)'}")
print(f"CRs         {CRS}")
print(f"lambda      {LAM_HEADLINE:g}   depth {DEPTH} bits")
print(f"patch       {INPUT_LEN} samples, overlap {PATCH_OVERLAP}")

## Model code

Inlined verbatim from `LARA_DAS_Validation_26102026.ipynb` (cell 4): `class LARA`,
`SharedDeltaCodec`, `make_laraq2`, `build_model` and the two baselines.  Nothing
is imported from another notebook at run time.

In [ ]:
"""The model code, inlined verbatim.  Nothing here is imported at run time.

Three sources, three jobs:

  Final_LARA_27092026.ipynb   class LARA, the two published baselines and the
                              AE_PureConcat architecture table
  LARA_Q_12102026.ipynb       the codec mixins, make_laraq2 and QuantizedBaseline

`class LARA` alone is not enough.  LARA-Q2 is a *different class*: make_laraq2
mixes SharedDeltaCodec and GatedEncoder into LARA's bases, so its state_dict has
gate_p, prior.* and _calib_span_t keys that no LARA ever had.  Loading a study
checkpoint into a bare LARA silently loads nothing but the trunk.
"""

# --------------------------------------------------------------------------
# from Final_LARA_27092026.ipynb -- LARA builds its optional
# variants from AttentionBlock / EnhancedResidualBlock, so these two
# come along even though pyramid_funnel32 does not use them.
# --------------------------------------------------------------------------

class AttentionBlock(nn.Module):
    """Simplified channel attention mechanism - EXACT ABLATION VERSION"""
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, max(4, channels//8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(max(4, channels//8), channels, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights

class EnhancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        
        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.elu(out)
        out = self.attention(out)
        return out

model_2_architecture = {
    100: {"layers": 2, "feature_maps": [16, 16]},
     60: {"layers": 2, "feature_maps": [16, 16]},
     50: {"layers": 2, "feature_maps": [16, 16]},
     30: {"layers": 2, "feature_maps": [8,  8 ]},
     20: {"layers": 2, "feature_maps": [16, 16]},
     15: {"layers": 2, "feature_maps": [16, 16]},
     10: {"layers": 2, "feature_maps": [16, 16]},
      5: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      3: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      2: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
}

class GeneralizedAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        latent_dim = 1500 // cr
        
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 375, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 375),
            nn.Unflatten(1, (32, 375)),
            nn.ConvTranspose1d(32, 16, kernel_size=9, stride=2, padding=4, output_padding=1), nn.ReLU(),
            nn.ConvTranspose1d(16, 1, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

class AE_PureConcat(nn.Module):
    def __init__(
        self,
        cr: int,
        input_len: int = 1500,
        kernel_size: int = 3,
        activation: nn.Module = nn.ELU,
        dropout: float = 0.1,
    ):
        super().__init__()
        cfg       = model_2_architecture[cr]
        layers    = cfg["layers"]
        fmaps     = cfg["feature_maps"]
        total_lat = input_len // cr

        # Distribute latent dimensions across branches
        base_dim  = total_lat // layers
        rem       = total_lat % layers
        latent_sizes = [base_dim + (1 if i < rem else 0) for i in range(layers)]

        # Encoder branches
        self.branches = nn.ModuleList()
        for i in range(layers):
            branch = nn.Sequential(
                nn.Conv1d(1, fmaps[i], kernel_size, padding=1),
                activation(inplace=True),
                nn.Dropout(dropout),
                nn.Flatten(),
                nn.Linear(input_len * fmaps[i], latent_sizes[i]),
                activation(inplace=True),
            )
            self.branches.append(branch)

        # Decoder: single linear layer to reconstruct full signal
        self.decoder = nn.Linear(total_lat, input_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, 1, input_len)
        zs = [branch(x) for branch in self.branches]     # list of (batch, latent_i)
        z  = torch.cat(zs, dim=1)                        # (batch, total_lat)
        out = self.decoder(z)                            # (batch, input_len)
        return out.unsqueeze(1)                          # → (batch, 1, input_len)

# --------------------------------------------------------------------------
# from Final_LARA_27092026.ipynb -- the trained trunk.  Self-contained:
# only nn / torch / np / math.
# --------------------------------------------------------------------------

class LARA(nn.Module):
    """
    LARA - Light Autoencoder for Reconstruction & Analysis.

    Single production configuration: variant='pyramid_funnel32' (see VARIANT in the CONFIG
    cell). Deep residual + channel-attention encoder, filled projection funnel, learned
    channel-pyramid decoder, conv refinement head.

    variant='pyramid_funnel32' (LARA): FILLS THE FUNNEL - T = L (no min(latent, L) clamp),
                               so the projection width no longer collapses as CR grows, and
                               BOTTLE_CHANNELS=32 -> 6016 projection columns (cr<=30) /
                               3008 (cr>30). cr<10 is identical to 'pyramid'.
    Compression ratio is preserved exactly: latent == input_len // cr (asserted).
    MSE loss only -> fair against the verbatim baseline recipe.

    Other internal tokens kept for provenance / ablations:
    variant='pyramid'   : single latent path; channel pyramid (16->32->64) + refinement head.
    variant='pyramid_v1': exact original 'pyramid' behaviour (archival alias).
    variant='pyramid_b32'    : 32-channel bottleneck spine for cr>=10 (16 for cr<10).
    variant='pyramid_b32_hd' : b32 + high-CR decoder polish for cr>=30 (ceiling 96, k9).
    variant='multiscale': latent split coarse (ceil half) + detail (rest), finest feature
                          carried in its own code, both decoded and fused.
    variant='pyramid_funnel64': same T = L fix with B=64 (cr<=30) / 128 (cr>30) -> 12032
                               columns at EVERY CR (baseline-parity width).
    """

    BOTTLE_CHANNELS = 16

    def __init__(self, cr, input_len=1500, variant="pyramid"):
        super().__init__()
        self.cr = cr
        self.input_len = input_len
        self.variant = variant
        self.latent_dim = input_len // cr

        # ---- variant knobs ----
        b32 = variant in ("pyramid_b32", "pyramid_b32_hd") and cr >= 10
        hd = variant == "pyramid_b32_hd" and cr >= 30
        funnel = variant in ("pyramid_funnel32", "pyramid_funnel64") and cr >= 10
        if variant == "pyramid_funnel32" and funnel:
            self.BOTTLE_CHANNELS = 32
        elif variant == "pyramid_funnel64" and funnel:
            self.BOTTLE_CHANNELS = 64 if cr <= 30 else 128
        else:
            self.BOTTLE_CHANNELS = 32 if b32 else 16
        self.pyramid_ceiling = max(64, self.BOTTLE_CHANNELS) if funnel else (96 if hd else 64)
        self.final_kernel = 9 if hd else 5

        self.L = 188 if cr <= 30 else 94
        self.T = self.L if funnel else min(self.latent_dim, self.L)

        if variant == "multiscale":
            self.latent_coarse = (self.latent_dim + 1) // 2
            self.latent_detail = self.latent_dim - self.latent_coarse
            self.T2 = min(32, max(2, self.latent_dim // 8))
        else:
            self.latent_coarse = self.latent_dim
            self.latent_detail = 0
            self.T2 = None
        assert self.latent_coarse + self.latent_detail == self.latent_dim, "CR must stay exact"

        # ---- encoder stages ----
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ELU(inplace=True),
            AttentionBlock(32),
        )
        self.stage1 = EnhancedResidualBlock(32, 64, stride=2, use_attention=True)
        self.stage2 = EnhancedResidualBlock(64, 128, stride=2, use_attention=True)
        self.stage3 = EnhancedResidualBlock(128, 256, stride=2, use_attention=True) if cr > 30 else None
        final_channels = 256 if self.stage3 is not None else 128

        # ---- coarse bottleneck (original Light path) ----
        self.bot_conv = nn.Sequential(
            nn.Conv1d(final_channels, self.BOTTLE_CHANNELS, kernel_size=1),
            nn.ELU(inplace=True),
        )
        self.bot_pool = nn.Sequential(
            nn.AdaptiveAvgPool1d(self.T),
            nn.Flatten(),
        )
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_coarse)

        # ---- detail bottleneck (multiscale only): finest 32ch @ 750 feature ----
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        if variant == "multiscale":
            self.detail_block = nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=3, padding=1),
                nn.BatchNorm1d(32),
                nn.ELU(inplace=True),
                nn.AdaptiveAvgPool1d(self.T2),
                nn.Flatten(),
            )
            self.detail_head = nn.Linear(32 * self.T2, self.latent_detail)

        # ---- decoders (channel pyramid + learned upsampling) ----
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_coarse, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)),
            nn.ELU(inplace=True),
            self._pyramid(self.BOTTLE_CHANNELS, self.T),
        )
        self.detail_decoder = nn.Sequential()
        if variant == "multiscale":
            self.detail_decoder = nn.Sequential(
                nn.Linear(self.latent_detail, self.BOTTLE_CHANNELS * self.T2),
                nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T2)),
                nn.ELU(inplace=True),
                self._pyramid(self.BOTTLE_CHANNELS, self.T2),
            )

        # ---- refinement head ----
        in_ch = 128 if variant == "multiscale" else self.pyramid_ceiling
        self.head = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=5, padding=2),
            nn.ELU(inplace=True),
            nn.Conv1d(32, 1, kernel_size=15, padding=7),
            nn.Sigmoid(),
        )

        self.test_dimensions()

    # ------------------------------------------------------------------ #
    # encoding / decoding
    # ------------------------------------------------------------------ #
    def encoder(self, x):
        """Concatenated latent (coarse + detail) - the actual transmitted code."""
        zc = self.encode_coarse(x)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            return torch.cat([zc, zd], dim=-1)
        return zc

    def encode_coarse(self, x):
        h = self.stage0(x)
        h = self.stage1(h)
        h = self.stage2(h)
        if self.stage3 is not None:
            h = self.stage3(h)
        f = self.bot_conv(h)
        f = self.bot_pool(f)
        return self.coarse_head(f)

    def encode_detail(self, x):
        f = self.stage0(x)
        d = self.detail_block(f)
        return self.detail_head(d)

    def forward(self, x):
        zc = self.encode_coarse(x)
        cf = self.coarse_decoder(zc)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            df = self.detail_decoder(zd)
            fused = torch.cat([cf, df], dim=1)
            return self.head(fused)
        return self.head(cf)

    # ------------------------------------------------------------------ #
    # helpers
    # ------------------------------------------------------------------ #
    def _pyramid(self, start_channels, start_len):
        """Channel-pyramid upsampler from start_len to input_len using learned
        (nearest + conv) upsampling. Channel ramp up to the variant ceiling."""
        nlev = 6
        ratio = self.input_len / start_len
        levels = []
        cch = start_channels
        prev_len = start_len
        for k in range(nlev):
            tgt = int(round(start_len * (ratio ** ((k + 1) / nlev))))
            tgt = max(tgt, prev_len + 1)
            if k == nlev - 1:
                tgt = self.input_len
            ksize = self.final_kernel if k == nlev - 1 else 5
            nc = min(self.pyramid_ceiling, int(cch * 2))
            levels.append(nn.Sequential(
                nn.Upsample(size=tgt, mode='nearest'),
                nn.Conv1d(cch, nc, kernel_size=ksize, padding=ksize // 2, bias=False),
                nn.BatchNorm1d(nc),
                nn.ELU(inplace=True),
            ))
            cch = nc
            prev_len = tgt
        return nn.Sequential(*levels)

    def test_dimensions(self):
        try:
            with torch.no_grad():
                test_input = torch.randn(1, 1, self.input_len)
                zc = self.encode_coarse(test_input)
                assert zc.numel() == self.latent_coarse, f"coarse latent {zc.numel()} != {self.latent_coarse}"
                if self.variant == "multiscale":
                    zd = self.encode_detail(test_input)
                    assert zd.numel() == self.latent_detail, f"detail latent {zd.numel()} != {self.latent_detail}"
                total = self.encoder(test_input).numel()
                assert total == self.latent_dim, f"total latent {total} != {self.latent_dim}"
                decoded = self(test_input)
                assert decoded.shape[2] == self.input_len, f"output {decoded.shape[2]} != {self.input_len}"
                actual_cr = self.input_len / total
                print(f"[OK] Light-HARA-V2 ({self.variant}) CR={self.cr}: Input {self.input_len} -> Latent {total} "
                      f"(coarse {self.latent_coarse} + detail {self.latent_detail}) -> Output {decoded.shape[2]} "
                      f"(True CR = {actual_cr:.2f})")
        except Exception as e:
            print(f"[FAIL] Light-HARA-V2 ({self.variant}) CR={self.cr}: {e}")
            self.create_fallback_architecture()

    def create_fallback_architecture(self):
        print("Creating fallback architecture...")
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4), nn.ELU(),
        )
        self.stage1 = nn.Identity()
        self.stage2 = nn.Identity()
        self.stage3 = None
        self.bot_conv = nn.Conv1d(64, self.BOTTLE_CHANNELS, kernel_size=1)
        self.bot_pool = nn.Sequential(nn.AdaptiveAvgPool1d(self.T), nn.Flatten())
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim)
        self.latent_coarse = self.latent_dim
        self.latent_detail = 0
        self.variant = "pyramid"
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)), nn.ELU(),
            nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7), nn.Sigmoid(),
        )
        self.detail_decoder = nn.Sequential()
        self.head = nn.Identity()
        print("[OK] Fallback architecture created")

# --------------------------------------------------------------------------
# from LARA_Q_12102026.ipynb -- the codec layer.
# --------------------------------------------------------------------------

class UniformQuantizer(nn.Module):
    """Single shared step size.  Additive noise in train, rounding in eval.

    Delta is a scalar Parameter so a trained step costs one number, and its
    gradient reaches it through the additive-noise path -- no straight-through
    estimator, because the training path is already linear in Delta.
    """

    def __init__(self, delta_init: float = 1.0, learn_delta: bool = False):
        super().__init__()
        if not (delta_init > 0):
            raise ValueError("delta_init must be positive")
        self.delta = nn.Parameter(torch.tensor(float(delta_init)),
                                  requires_grad=bool(learn_delta))

    def indices(self, z):
        """Integer symbol index per code position.  This is what gets coded."""
        return torch.round(z / self.delta)

    def dequantize(self, idx):
        return idx * self.delta

    def forward(self, z):
        if self.training:
            return z + self.delta * torch.empty_like(z).uniform_(-0.5, 0.5)
        return self.dequantize(self.indices(z))

    def extra_repr(self):
        return f"delta={float(self.delta.detach()):.6g}, learn_delta={self.delta.requires_grad}"

def _laplace_cdf(x, scale):
    s = scale.clamp_min(1e-9)
    xa = (x / s).clamp(-30.0, 30.0)
    return torch.where(xa <= 0, 0.5 * torch.exp(xa), 1.0 - 0.5 * torch.exp(-xa))

class CausalLaplacePrior(nn.Module):
    """Scale-only context model over the code positions.

    z has shape (B, d), read as a length-d, one-channel sequence.  Position i's
    scale depends on z_{i-K..i-1} ONLY, enforced by a left-padded causal
    convolution -- there is no path from z_i to s_i, which is what makes the
    rate non-trivial.  p(z_i) is the Laplace mass integrated across that
    position's bin, so the training rate is the rate the matching range coder
    actually achieves.
    """

    def __init__(self, n_ctx: int = 16, kernel: int = 5):
        super().__init__()
        if kernel < 2:
            raise ValueError("kernel must be >= 2 for a context to exist")
        self.kernel = int(kernel)
        self.n_ctx = int(n_ctx)
        self.ctx = nn.Conv1d(1, n_ctx, kernel_size=kernel)
        self.head = nn.Conv1d(n_ctx, 1, kernel_size=1)

    def scales(self, z):
        h = self.ctx(F.pad(z.unsqueeze(1), (self.kernel - 1, 0)))
        return F.softplus(self.head(h)).squeeze(1) + 1e-6

    def bin_prob(self, z_q, scales, delta):
        half = 0.5 * delta
        return (_laplace_cdf(z_q + half, scales) - _laplace_cdf(z_q - half, scales)).clamp_min(1e-9)

    def rate_bits(self, z, delta, floor_bits: float = 1e-9):
        s = self.scales(z)
        p = self.bin_prob(z, s, delta)
        return (-torch.log2(p).sum(dim=1)).clamp_min(floor_bits)

class SharedDeltaCodec:
    """Shared-scalar Delta calibration.

    Delta is ONE scalar: 16 bits per model.  Per-dimension float16 ranges cost
    2*d*16 bits -- 2400 bits at CR=20 against a 265-bit payload -- so a
    per-dimension convention would compare bookkeeping, not codecs.
    """

    _q_armed = False

    def quantize(self, z):
        if not self._q_armed:
            return z
        return self.quantizer(z)

    def bits_per_symbol(self):
        """Depth implied by the current step size.

        The alphabet is SYMMETRIC: depth b holds indices -2**(b-1) .. 2**(b-1)-1,
        spanning 2**b steps of width Delta and reaching +-2**(b-1)*Delta.  The
        calibrated span S is a max |z|, i.e. a HALF-width, so

            h = round(S / Delta)   positive levels
            b = ceil(log2(h + 1)) + 1

        Getting this factor of two wrong makes Delta twice as coarse as the
        nominal depth claims, and every symbol past +-2**(b-1) overflows.
        """
        span = float(self._calib_span_t.detach())
        if span <= 0:
            raise RuntimeError("step size not calibrated: call set_calibrated_delta(z)")
        h = int(round(span / float(self.quantizer.delta.detach())))
        return max(1, int(math.ceil(math.log2(h + 1))) + 1)

    def calibrate_prior(self, z_calib):
        """Bias the prior head to the calibrated latent scale.

        Without this the head starts at softplus(0)=0.693 while the latent is
        O(1e-2), so the discretised Laplace is flat across the bin: every symbol
        costs the same, the rate is identical for every trace, and the context
        contributes nothing.  Only head.bias is touched -- zeroing the conv would
        kill the gradient into it.
        """
        if getattr(self, "prior", None) is None:
            return None
        with torch.no_grad():
            target = max(float(z_calib.detach().float().abs().mean().item()), 1e-6)
            inv = math.log(math.expm1(target)) if target < 20.0 else target
            self.prior.head.bias.fill_(float(inv))
        return target

    def set_calibrated_delta(self, z_calib, target_bits=None):
        """Fit the shared step size on TRAINING latents only.  Never on val or test."""
        with torch.no_grad():
            z = z_calib.detach().float().reshape(-1)
            span = float(z.abs().max().item())
            self._calib_span_t.fill_(span)
            if target_bits is not None:
                half_levels = max(1, (1 << (int(target_bits) - 1)) - 1)
                self.quantizer.delta.data.fill_(float(span / half_levels))
            self.calibrate_prior(z_calib)
        return (float(self.quantizer.delta.detach()), self.bits_per_symbol(),
                float(self._calib_span_t.detach()))

    def rate_bits(self, z_q):
        """Expected rate in bits for an ALREADY quantized latent."""
        d = float(z_q.shape[-1])
        if getattr(self, "prior", None) is None:
            return torch.full((z_q.shape[0],), d * self.bits_per_symbol(),
                              device=z_q.device, dtype=z_q.dtype)
        return self.prior.rate_bits(z_q, self.quantizer.delta)

class LaraTrunk:
    """class LARA's conv trunk, down to the flattened B*T feature vector."""

    def _features_impl(self, x):
        h = self.stage0(x)
        h = self.stage1(h)
        h = self.stage2(h)
        if self.stage3 is not None:
            h = self.stage3(h)
        return self.bot_pool(self.bot_conv(h))

class SparseCode(nn.Module):
    """B*T quantized values pruned to the d largest-magnitude ones.

    Straight-through top-k: the mask is hard forward (so training and coding see
    the same code) and identity backward (so the encoder still gets gradient).
    Per trace, so one mask per sample.
    """

    def __init__(self, d):
        super().__init__()
        self.d = int(d)

    def forward(self, f):
        """f: (B, N) -> (masked (B, N), survivor indices (B, k))."""
        B, N = f.shape
        k = min(self.d, N)
        if k == N:
            idx = torch.arange(N, device=f.device).expand(B, N)
            return f, idx
        idx = f.abs().topk(k, dim=1).indices
        mask = torch.zeros_like(f).scatter_(1, idx, 1.0)
        out = f * mask + f * (1.0 - mask) if self.training else f * mask
        return out, idx

class GatedEncoder:
    """Identity-initialised elementwise gate on the flattened B*T feature vector.

    LARA's AttentionBlock re-weights *channels* through AdaptiveAvgPool1d(1) and
    never touches the flattened pre-projection vector, so it cannot express
    "this (channel, time) feature matters at this CR".
    """

    _gate_ready = False

    def _init_gate(self, ft, gate_mode="elem", gate_rank=64):
        if gate_mode not in ("elem", "lowrank", "none"):
            raise ValueError(f"unknown gate_mode {gate_mode!r}")
        self.gate_mode = gate_mode
        self.gate_rank = int(gate_rank)
        if gate_mode == "elem":
            self.gate_p = nn.Parameter(torch.zeros(int(ft)))
            self.gate_p_used = True
        elif gate_mode == "lowrank":
            self.gate_l1 = nn.Linear(int(ft), self.gate_rank)
            self.gate_l2 = nn.Linear(self.gate_rank, int(ft))
            nn.init.zeros_(self.gate_l2.weight)
            nn.init.zeros_(self.gate_l2.bias)
            self.gate_p = None
        else:
            self.gate_p = None
        self.feature_gate = nn.Identity()
        self.gate_dim = int(ft)
        # Arms the gate.  Without this the gate is never applied, the
        # "identity at init" check passes VACUOUSLY, and the parameter
        # silently receives no gradient -- so selftest() asserts the gate
        # is really in use, not merely present.
        self._gate_ready = True

    def _gate(self, f):
        if self.gate_mode == "elem":
            return f * (1.0 + torch.tanh(self.gate_p))
        if self.gate_mode == "lowrank":
            return f * (1.0 + torch.tanh(self.gate_l2(F.relu(self.gate_l1(f)))))
        return f

    def _features_impl(self, x):
        raise NotImplementedError

    def features(self, x):
        """Flattened, gated B*T feature vector.  The unit the gate acts on."""
        f = self._features_impl(x)
        return self._gate(f) if self._gate_ready else f

def make_laraq2(LARA):
    """LARA + feature gate + shared-Delta quantizer + causal Laplace prior."""

    # LaraTrunk must precede GatedEncoder in the bases: GatedEncoder defines
    # _features_impl as abstract, and Python resolves same-name methods
    # left-to-right, so the reverse order would shadow the real implementation.
    class LARAQ2(SharedDeltaCodec, LaraTrunk, GatedEncoder, LARA):
        def __init__(self, cr, input_len=INPUT_LEN, variant=LARA_VARIANT,
                     gate_mode="elem", gate_rank=64, delta_init=1.0,
                     learn_delta=False, prior_ctx=16, prior_kernel=5,
                     use_prior=True):
            super().__init__(cr, input_len=input_len, variant=variant)
            self._init_gate(self.BOTTLE_CHANNELS * self.T, gate_mode, gate_rank)
            self.quantizer = UniformQuantizer(delta_init, learn_delta)
            self.register_buffer("_calib_span_t", torch.zeros(()))
            self.prior = CausalLaplacePrior(prior_ctx, prior_kernel) if use_prior else None
            self.arch = 2
            self._q_armed = True
            self.test_dimensions()

        def encode_coarse(self, x):
            return self.coarse_head(self.features(x))

        def decode_latent(self, z_q):
            return self.head(self.coarse_decoder(z_q))

        def forward(self, x):
            return self.decode_latent(self.quantize(self.encode_coarse(x)))

        def describe(self):
            return _describe(self)

    return LARAQ2

# --------------------------------------------------------------------------
# from LARA_Q_12102026.ipynb -- the baseline adapter.  The base.
# prefix on GeneralizedAutoencoder / AE_PureConcat weights comes
# from this wrapper.
# --------------------------------------------------------------------------

def baseline_encode(model, name, x):
    """The code vector each baseline actually transmits."""
    if name == "AE_PureConcat":
        return torch.cat([b(x) for b in model.branches], dim=1)
    return model.encoder(x)

def baseline_decode(model, name, z):
    y = model.decoder(z)
    return y if y.dim() == 3 else y.unsqueeze(1)

def latent_dim_of(base, cr, input_len=INPUT_LEN):
    """Code length of a baseline.

    `class LARA` and `Light_Capacity_HARA` store `self.latent_dim`, but
    `GeneralizedAutoencoder` and `AE_PureConcat` keep it as a *local* in
    __init__ and never attach it, so attribute access on them raises
    AttributeError.  Fall back to the definition both classes use.
    """
    d = getattr(base, "latent_dim", None)
    return int(d) if d else int(input_len) // int(cr)

class QuantizedBaseline(SharedDeltaCodec, nn.Module):
    """Adapter putting a published architecture behind the codec contract.

    Inheriting SharedDeltaCodec is what lets the *same* trainer, the same
    calibration and the same evaluation run for every series -- one code path,
    not five.
    """

    def __init__(self, base, name, cr, input_len=INPUT_LEN, delta_init=1.0,
                 learn_delta=False):
        super().__init__()
        self.base = base
        self.name = name
        self.cr = int(cr)
        self.input_len = input_len
        self.latent_dim = latent_dim_of(base, cr, input_len)
        self.quantizer = UniformQuantizer(delta_init, learn_delta)
        self.register_buffer("_calib_span_t", torch.zeros(()))
        self.prior = None                      # published baselines have no prior
        self.arch = 2
        self._q_armed = True

    def encode_coarse(self, x):
        return baseline_encode(self.base, self.name, x)

    def decode_latent(self, z_q):
        return baseline_decode(self.base, self.name, z_q)

    def forward(self, x):
        return self.decode_latent(self.quantize(self.encode_coarse(x)))

    def describe(self):
        return {"name": self.name, "latent_dim": self.latent_dim,
                "B_times_T": int(self.latent_dim), "gate_mode": "none",
                "gate_params": 0, "quantizer_params": 1, "prior_params": 0,
                "total_params": sum(p.numel() for p in self.parameters())}



# ---------------------------------------------------------------- factories
# build_baseline_base is rewritten: the study notebook reaches into a sibling
# notebook for these two classes, which the GPU host does not have.  The classes
# themselves are the study's, copied above.


def build_baseline_base(name, cr):
    """Instantiate a published baseline architecture, untrained."""
    if name == "GeneralizedAutoencoder":
        return GeneralizedAutoencoder(cr)
    if name == "AE_PureConcat":
        return AE_PureConcat(cr, INPUT_LEN)
    raise ValueError(f"unknown baseline {name!r}")


def class_for(series):
    """The trained class for a series name, exactly as the study built it."""
    if series == "LARA-Q2":
        return make_laraq2(LARA)
    if series in BASELINE_NAMES:
        return None                      # wrapped by QuantizedBaseline instead
    raise ValueError(f"unknown series {series!r}")


def build_model(series, cr, name=None):
    """Construct an untrained model of `series` at compression ratio `cr`."""
    if series == "LARA-Q2":
        return make_laraq2(LARA)(cr, variant=LARA_VARIANT, gate_mode="elem")
    name = name or series
    base = build_baseline_base(name, cr)
    return QuantizedBaseline(base, name, cr, delta_init=1.0, learn_delta=False)


print(f"inlined: LARA, GeneralizedAutoencoder, AE_PureConcat, "
      f"LARAQ2 = make_laraq2(LARA), QuantizedBaseline")

## Codec helpers, patching and overlap-add

`code_and_decode_patches` does exactly what the validated measurement did per
window: per-patch min-max, `encode_coarse`, uniform quantisation with the
calibrated shared Delta, `decode_latent`, then restore the patch amplitude.

In [ ]:
def nominal_alphabet(bits):
    """Symmetric alphabet of exactly 2**bits symbols, centred on zero."""
    half = 1 << (bits - 1)
    return -half, half - 1, 1 << bits, half


def clamp_to_alphabet(indices, bits):
    lo, hi, A, off = nominal_alphabet(bits)
    n_out = int(((indices < lo) | (indices > hi)).sum())
    return np.clip(indices, lo, hi), n_out, A, off


def per_trace_snr(x, y, eps=1e-12):
    """Per-trace SNR, then the mean over finite values."""
    out = []
    for i in range(x.shape[0]):
        a = np.asarray(x[i], dtype=np.float64).reshape(-1)
        b = np.asarray(y[i], dtype=np.float64).reshape(-1)
        mse = float(np.mean((a - b) ** 2))
        p = float(np.mean(a ** 2))
        out.append(10.0 * math.log10(p / (mse + eps))
                   if mse > 0 and p > 0 else float("inf"))
    v = np.asarray(out, dtype=float)
    return float(np.mean(v[np.isfinite(v)])), v


def window_starts(n_samples, window=INPUT_LEN, overlap=PATCH_OVERLAP):
    """Patch starts; the final patch is aligned with the final sample."""
    if n_samples <= window:
        return [0]
    stride = window - overlap
    starts = list(range(0, n_samples - window + 1, stride))
    final = n_samples - window
    if starts[-1] != final:
        starts.append(final)
    return starts


def gaussian_weights(length=INPUT_LEN, sigma_fraction=GAUSS_SIGMA_FRACTION,
                     floor=GAUSS_FLOOR):
    x = np.arange(length, dtype=np.float64)
    centre = (length - 1) / 2.0
    sigma = sigma_fraction * length
    w = np.exp(-0.5 * ((x - centre) / sigma) ** 2)
    w /= w.max()
    return np.maximum(w, floor)


def code_and_decode_patches(patches, model, delta, bits):
    """(N, 1500) amplitude patches -> (N, 1500) reconstructions."""
    out = np.empty_like(patches, dtype=np.float64)
    n_clamped = 0
    for i in range(0, len(patches), BATCH):
        block = np.asarray(patches[i:i + BATCH], dtype=np.float64)
        mn = block.min(axis=1, keepdims=True)
        mx = block.max(axis=1, keepdims=True)
        rng = mx - mn
        nz = rng[:, 0] > 1e-8
        xn = np.zeros_like(block)
        xn[nz] = (block[nz] - mn[nz]) / rng[nz]

        x = torch.tensor(xn[:, None, :], dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            z = model.encode_coarse(x)
            idx = model.quantizer.indices(z).cpu().numpy().astype(np.int64)
        idx, nc, _, _ = clamp_to_alphabet(idx, bits)
        n_clamped += nc

        z_q = torch.as_tensor(idx, dtype=torch.float32, device=DEVICE) * delta
        with torch.no_grad():
            y = model.decode_latent(z_q)
        yn = y.cpu().numpy()[:, 0, :].astype(np.float64)
        yn = yn * rng + mn
        yn[~nz] = mn[~nz]
        out[i:i + BATCH] = yn
    return out, n_clamped


def reconstruct_2d(data, model, delta, bits):
    """Patch every channel, decode, and overlap-add back to the full trace."""
    n_ch, n_t = data.shape
    starts = window_starts(n_t)
    weights = gaussian_weights(INPUT_LEN)

    patches = []
    for ch in range(n_ch):
        tr = np.asarray(data[ch], dtype=np.float32).copy()
        bad = ~np.isfinite(tr)
        if bad.any():
            good = tr[~bad]
            fill = float(np.median(good)) if good.size else 0.0
            tr[bad] = fill
        for s in starts:
            patches.append(tr[s:s + INPUT_LEN])
    patches = np.stack(patches)

    recs, n_clamped = code_and_decode_patches(patches, model, delta, bits)

    rec = np.empty_like(data, dtype=np.float32)
    k = 0
    for ch in range(n_ch):
        acc = np.zeros(n_t, dtype=np.float64)
        wsum = np.zeros(n_t, dtype=np.float64)
        for s in starts:
            acc[s:s + INPUT_LEN] += recs[k] * weights
            wsum[s:s + INPUT_LEN] += weights
            k += 1
        rec[ch] = (acc / np.maximum(wsum, 1e-12)).astype(np.float32)
    return rec, n_clamped, starts


def robust_limit(a, percentile=DISPLAY_PERCENTILE):
    v = np.asarray(a)[np.isfinite(a)]
    if v.size == 0:
        return 1.0
    lim = float(np.percentile(np.abs(v), percentile))
    if not np.isfinite(lim) or lim <= 0:
        lim = float(np.max(np.abs(v)))
    return lim if np.isfinite(lim) and lim > 0 else 1.0

## Load the DAS records and calibrate Delta

The calibration pool is rebuilt exactly as in the validation run: global
channel index modulo 10, residues 0-7, shuffle with seed 42, first 2000
channels, first 1500 samples, per-trace min-max.  The images use the full 2000
samples of the chosen event.

In [ ]:
def _orient(arr):
    """Return (channels, time) with the 960-channel axis identified."""
    if arr.ndim != 2:
        return None
    n0, n1 = arr.shape
    if n0 == DAS_EXPECTED_TRACES and n1 >= INPUT_LEN:
        return arr
    if n1 == DAS_EXPECTED_TRACES and n0 >= INPUT_LEN:
        return arr.T
    if n0 < n1 and n1 >= INPUT_LEN:
        return arr
    if n1 < n0 and n0 >= INPUT_LEN:
        return arr.T
    return None


EVENT_ARRAYS = {}
EVENT_FILES = {}
for ev in DAS_EVENTS_ALL:
    path = os.path.join(DAS_DIR, f"out_eq_{ev}_pre.mat")
    if not os.path.isfile(path):
        print(f"[das] event {ev}: not found ({path})")
        continue
    try:
        with h5py.File(path, "r") as f:
            if DAS_KEY not in f:
                raise KeyError(f"key {DAS_KEY!r} missing; have {list(f.keys())}")
            raw = np.asarray(f[DAS_KEY], dtype=np.float32)
    except Exception as exc:
        print(f"[das] event {ev}: read failed ({exc!r})")
        continue
    arr = _orient(raw)
    if arr is None:
        print(f"[das] event {ev}: shape {raw.shape} rejected")
        continue
    EVENT_ARRAYS[ev] = arr
    EVENT_FILES[ev] = path
    print(f"[das] event {ev}: {raw.shape} -> {arr.shape}  (channels, samples)")

if not EVENT_ARRAYS:
    raise RuntimeError(
        "no DAS event could be loaded.  Set LARA_DAS_DIR to the folder that "
        "contains out_eq_1_pre.mat, out_eq_44_pre.mat and out_eq_74_pre.mat.")

if DAS_EVENT not in EVENT_ARRAYS:
    fallback = sorted(EVENT_ARRAYS)[0]
    print(f"[das] event {DAS_EVENT} unavailable; showing event {fallback}")
    DAS_EVENT = fallback

# ------------------------------------------------------------ calibration pool
calib_raw = []
global_index = 0
for ev in sorted(EVENT_ARRAYS):
    arr = EVENT_ARRAYS[ev]
    for ch in range(arr.shape[0]):
        role = global_index % SPLIT_STRIDE
        global_index += 1
        if role not in CALIB_RESIDUES:
            continue
        seg = np.asarray(arr[ch, WINDOW_START:WINDOW_START + INPUT_LEN],
                         dtype=np.float32)
        if seg.size != INPUT_LEN or not np.isfinite(seg).all():
            continue
        if float(seg.max() - seg.min()) <= 0:
            continue
        calib_raw.append(seg)

_rng = random.Random(SEED)
_rng.shuffle(calib_raw)
calib_raw = calib_raw[:min(N_CALIB_TARGET, len(calib_raw))]

CALIB_NORM = []
for seg in calib_raw:
    lo, hi = float(seg.min()), float(seg.max())
    CALIB_NORM.append((seg - lo) / (hi - lo))

print(f"[calib] {len(CALIB_NORM)} calibration channels "
      f"(target {N_CALIB_TARGET})")

# ------------------------------------------------------------ data to display
PLOT = np.asarray(EVENT_ARRAYS[DAS_EVENT], dtype=np.float32)
if MAX_CHANNELS > 0:
    PLOT = PLOT[:min(MAX_CHANNELS, PLOT.shape[0])]
    print(f"[plot] limited to the first {PLOT.shape[0]} channels "
          f"(LARA_Q2_CHANNELS={MAX_CHANNELS})")
TRACE_LEN = PLOT.shape[1]
N_CHANNELS = PLOT.shape[0]
print(f"[plot] event {DAS_EVENT}: {N_CHANNELS} channels x {TRACE_LEN} samples")

## Load the six LARA-Q2 checkpoints

Only LARA-Q2 at `lambda = 1e-4`.  The shared Delta is calibrated on the
calibration latents at 10 bits, exactly as in the validated run, and never on
the channels being displayed.

In [ ]:
def _lam_tag(lam):
    return (f"{lam:g}".replace("+", "")
            .replace(".", "p").replace("e-", "em").replace("-", "m"))


def calib_latents(model, batch=64):
    zs = []
    for i in range(0, len(CALIB_NORM), batch):
        seg = np.stack(CALIB_NORM[i:i + batch])
        x = torch.tensor(seg[:, None, :], dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            zs.append(model.encode_coarse(x).reshape(-1).cpu())
    return torch.cat(zs)


MODELS = {}
for cr in CRS:
    width = int(round(INPUT_LEN / cr))
    tag = f"LARA-Q2_d{width}_lam{_lam_tag(LAM_HEADLINE)}"
    path = os.path.join(CKPT_DIR, f"best_{tag}.pth")
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"checkpoint not found: {path}\n"
            "Set LARA_CKPT_DIR or LARA_RESULTS_DIR so that "
            "ckpts_baselines/best_LARA-Q2_d*.pth is reachable.")

    model = build_model("LARA-Q2", cr)
    blob = torch.load(path, map_location="cpu", weights_only=False)
    state = blob.get("state_dict", blob)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE)
    model.eval()

    z_calib = calib_latents(model)
    delta, depth, span = model.set_calibrated_delta(z_calib, target_bits=DEPTH)
    if int(depth) != int(DEPTH):
        raise RuntimeError(f"CR {cr}: calibrated depth {depth} != {DEPTH}")

    MODELS[cr] = {"model": model, "delta": float(delta), "depth": int(depth),
                  "width": width, "tag": tag, "path": path}
    print(f"[ckpt] CR {cr:3d}  d={width:3d}  delta={delta:.3e}  "
          f"span={span:.4f}  {os.path.basename(path)}")

## Reconstruct every CR

Two patches per 2000-sample trace (starts 0 and 500), so the effective coded
rate is 1.5x the per-patch rate.  The per-patch rate is read from the validated
`coded_rates` CSV of the same checkpoint; if the CSV is not present the nominal
`10 bits x width / 1500` is used and labelled as such.

In [ ]:
def validated_rate(width):
    """Best validated point of this checkpoint (highest SNR, then lowest bps)."""
    tag = f"LARA-Q2_d{width}_lam{_lam_tag(LAM_HEADLINE)}"
    path = os.path.join(VAL_DIR, "coded_rates", tag + ".csv")
    if not os.path.isfile(path):
        return None, None, "nominal"
    rows = [r for r in csv.DictReader(open(path, encoding="utf-8"))
            if r.get("series")]
    best = None
    for r in rows:
        try:
            b = float(r["bps_total_amortised"])
            s = float(r["snr_db"])
        except (KeyError, TypeError, ValueError):
            continue
        if not np.isfinite(b) or not np.isfinite(s):
            continue
        if best is None or s > best[1] + 1e-9 or (
                abs(s - best[1]) <= 1e-9 and b < best[0]):
            best = (b, s)
    if best is None:
        return None, None, "nominal"
    return best[0], best[1], "validated"


RESULTS = {}
for cr in CRS:
    info = MODELS[cr]
    t0 = time.time()
    rec, n_clamped, starts = reconstruct_2d(
        PLOT, info["model"], info["delta"], info["depth"])
    resid = (PLOT.astype(np.float64) - rec.astype(np.float64)).astype(np.float32)

    snr, per_snr = per_trace_snr(PLOT, rec)
    mse = float(np.mean(resid.astype(np.float64) ** 2))
    flat_a = PLOT.reshape(-1).astype(np.float64)
    flat_b = rec.reshape(-1).astype(np.float64)
    corr = float(np.corrcoef(flat_a, flat_b)[0, 1]) if (
        np.std(flat_a) > 0 and np.std(flat_b) > 0) else 0.0

    bps_patch, snr_valid, rate_src = validated_rate(info["width"])
    if bps_patch is None:
        bps_patch = info["depth"] * info["width"] / INPUT_LEN
        rate_src = "nominal"
    bps_eff = bps_patch * (len(starts) * INPUT_LEN / TRACE_LEN)

    RESULTS[cr] = {
        "cr": cr, "width": info["width"], "depth": info["depth"],
        "delta": info["delta"], "bps_patch": bps_patch,
        "bps_effective": bps_eff, "rate_source": rate_src,
        "snr_db": snr, "snr_validated_window": snr_valid,
        "mse": mse, "correlation": corr, "n_clamped": n_clamped,
        "n_channels": N_CHANNELS, "n_patches_per_trace": len(starts),
        "recon": rec, "residual": resid, "per_trace_snr": per_snr,
    }
    print(f"[run] CR {cr:3d}  {bps_patch:7.4f} bps/patch  "
          f"{bps_eff:7.4f} bps effective  SNR {snr:6.2f} dB  "
          f"corr {corr:.4f}  clamped {n_clamped}  "
          f"({time.time() - t0:.1f} s)")

print(f"[run] patches per trace: {RESULTS[CRS[0]]['n_patches_per_trace']}  "
      f"effective rate factor {RESULTS[CRS[0]]['bps_effective'] / max(RESULTS[CRS[0]]['bps_patch'], 1e-12):.3f}x")

## 2D images: original, reconstructed, residual

One figure per CR.  `seismic` colormap, symmetric limits at the 99th percentile
of |amplitude|; the residual uses its own limits.

In [ ]:
EXTENT = [0, N_CHANNELS - 1, TRACE_LEN - 1, 0]
ORIG_LIMIT = robust_limit(PLOT)


def plot_cr(cr):
    res = RESULTS[cr]
    rlimit = robust_limit(res["residual"])
    fig, axes = plt.subplots(3, 1, figsize=(16, 18), sharex=True)

    panels = [
        (PLOT, "Original DAS data - event %d" % DAS_EVENT,
         "Amplitude", ORIG_LIMIT, axes[0]),
        (res["recon"],
         "LARA-Q2 CR=%d - reconstructed  (%.4f bps/patch, %.4f bps effective, SNR %.2f dB)"
         % (cr, res["bps_patch"], res["bps_effective"], res["snr_db"]),
         "Amplitude", ORIG_LIMIT, axes[1]),
        (res["residual"], "Reconstruction residual (original - reconstructed)",
         "Residual amplitude", rlimit, axes[2]),
    ]
    for img, title, cbar_label, lim, ax in panels:
        im = ax.imshow(np.asarray(img).T, cmap="seismic", aspect="auto",
                       interpolation="nearest", origin="upper",
                       vmin=-lim, vmax=lim, extent=EXTENT)
        ax.set_title(title)
        ax.set_ylabel("Time sample")
        ax.set_xlabel("Channel number")
        fig.colorbar(im, ax=ax, label=cbar_label, pad=0.01)

    fig.suptitle("DAS event %d - LARA-Q2, CR %d" % (DAS_EVENT, cr),
                 fontsize=16, y=0.995)
    fig.tight_layout()
    out = os.path.join(FIG_DIR, "CR%03d_original_reconstructed_residual.png" % cr)
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[fig] {out}")
    return out


FIGURES = [plot_cr(cr) for cr in CRS]

## Combined six-CR figure

Rows = CR, columns = original, reconstructed, residual.  The original column is
repeated because it is the same in every row.

In [ ]:
fig, axes = plt.subplots(len(CRS), 3, figsize=(20, 28), sharex=True)
for row, cr in enumerate(CRS):
    res = RESULTS[cr]
    rlimit = robust_limit(res["residual"])
    items = [
        (PLOT, ORIG_LIMIT, "Original (event %d)" % DAS_EVENT),
        (res["recon"], ORIG_LIMIT,
         "CR %d - recon  %.4f bps  SNR %.2f dB"
         % (cr, res["bps_patch"], res["snr_db"])),
        (res["residual"], rlimit, "CR %d - residual" % cr),
    ]
    for col, (img, lim, title) in enumerate(items):
        ax = axes[row, col]
        im = ax.imshow(np.asarray(img).T, cmap="seismic", aspect="auto",
                       interpolation="nearest", origin="upper",
                       vmin=-lim, vmax=lim, extent=EXTENT)
        ax.set_title(title, fontsize=11)
        if row == len(CRS) - 1:
            ax.set_xlabel("Channel number")
        if col == 0:
            ax.set_ylabel("Time sample")
        fig.colorbar(im, ax=ax, pad=0.01, fraction=0.03)

fig.suptitle("LARA-Q2 on DAS event %d - six CR values" % DAS_EVENT,
             fontsize=18, y=1.0)
fig.tight_layout()
COMBINED = os.path.join(FIG_DIR, "combined_6cr_original_reconstructed_residual.png")
fig.savefig(COMBINED, dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"[fig] {COMBINED}")

## Summary table and arrays

`summary.csv` holds one row per CR.  The NPZ stores the original and the six
reconstructions/residuals for the displayed event.

In [ ]:
SUMMARY = os.path.join(OUT_DIR, "summary.csv")
fields = ["cr", "width", "depth", "delta", "bps_patch", "bps_effective",
          "rate_source", "snr_db", "snr_validated_window", "mse",
          "correlation", "n_clamped", "n_channels", "n_patches_per_trace"]
with open(SUMMARY, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=fields)
    w.writeheader()
    for cr in CRS:
        r = RESULTS[cr]
        w.writerow({k: r[k] for k in fields})
print(f"[saved] {SUMMARY}")

if SAVE_NPZ:
    payload = {"original": PLOT.astype(np.float32),
               "event": np.array([DAS_EVENT]),
               "trace_len": np.array([TRACE_LEN])}
    for cr in CRS:
        payload["recon_cr%d" % cr] = RESULTS[cr]["recon"].astype(np.float32)
        payload["residual_cr%d" % cr] = RESULTS[cr]["residual"].astype(np.float32)
    npz = os.path.join(OUT_DIR, "reconstruction_eq%d.npz" % DAS_EVENT)
    np.savez_compressed(npz, **payload)
    print(f"[saved] {npz}")

print("\nCR    bps/patch  bps/eff   SNR dB   corr")
for cr in CRS:
    r = RESULTS[cr]
    print("%-4d  %8.4f  %8.4f  %7.2f  %6.4f"
          % (cr, r["bps_patch"], r["bps_effective"], r["snr_db"], r["correlation"]))